### Auf und Ab – Lösung Schritt für Schritt

*Jugendwettbewerb Informatik 2018, 3. Runde, Junioraufgabe 1 (= 37. Bundeswettbewerb Informatik, 1. Runde, Junioraufgabe 1)*

Frederick hat auf dem Speicher das Leiterspiel gefunden. Die Spielregeln sind einfach. Wer an der Reihe ist, würfelt einmal
und bewegt dann seine Spielfigur entsprechend der gewürfelten Augenzahl weiter. Kommt die Figur dabei an einem Leiterende
zu stehen, wird sie direkt zum anderen Ende der Leiter bewegt. Das gilt sowohl aufwärts als auch abwärts: Endet der Zug
z. B. auf Feld 6, geht es direkt zu Feld 27. Endet der Zug hingegen auf der 27, geht es leider zurück zur 6.

Gewonnen hat, wer mit seiner Figur zuerst das Zielfeld 100 exakt erreicht. Das bedeutet: Wird kurz vor dem Zielfeld eine zu
große Zahl gewürfelt, muss die Spielfigur den überschüssigen Rest zurücklaufen. Steht die Figur zum Beispiel auf der 99 und
ihr Spieler würfelt eine Vier, muss die Figur zur 97 bewegt werden.

Frederick überlegt sich: Wenn man nur Einsen würfelt, erreicht man auf jeden Fall das Ziel – aber bei mehreren Mitspielern
wohl nicht als Erster. Frederick will wissen, ob man auch für das wiederholte Würfeln eines der anderen Werte von Zwei bis
Sechs das Ziel erreicht.

#### Junioraufgabe 1

Du möchtest Frederick helfen. Schreibe ein Programm, das für alle sechs möglichen Zahlen herausfindet, ob man beim
Leiterspiel das Ziel erreicht, wenn man stets diese Zahl würfelt. Wie oft müsste Frederick jeweils würfeln, sofern er
überhaupt ins Ziel kommt?

Auf den BWINF-Webseiten ist das Spielbrett des Leiterspiels abgebildet. Diesem Bild kannst du entnehmen, welche Felder mit
Leitern verbunden sind.

#### Das Spielbrett

Das Spielbrett (`beispieldaten/leiterspiel.pdf`) hat die Felder 1 bis 100 und zwölf Leitern. Jede Leiter verbindet zwei Felder:

```
(6, 27), (14, 19), (21, 53), (31, 42), (33, 38), (46, 62),
(51, 59), (57, 96), (65, 85), (68, 80), (70, 76), (92, 98)
```

Gestartet wird auf Feld 0, außerhalb des Spielbretts.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Simulation:** Wir spielen das Spiel einfach nach, Zug für Zug. Ein Zug ist immer dieselbe Rechnung.
- **Wörterbuch (`dict`) für die Leitern:** `leitern[6]` ergibt 27. Mit `if feld in leitern` prüft man, ob auf dem Feld eine Leiter liegt.
- **Endlosschleife erkennen:** Würfelt man immer dieselbe Zahl, hängt der nächste Zug nur vom aktuellen Feld ab. Sieht man ein Feld zum
  **zweiten Mal**, wiederholt sich alles für immer. Dafür merkt man sich die besuchten Felder (in einer Liste oder Menge).
  Alternativ bricht man nach 100 Zügen ab: Es gibt nur 100 Felder, mehr Züge ohne Wiederholung sind nicht möglich.
- **Zustandsfolge als Graph:** Jedes Feld hat genau einen Nachfolger. Der Weg endet entweder im Ziel oder läuft in einen Kreis.
  (Verfahren von Floyd, „Hase und Igel“, findet solche Kreise ohne Speicher, hier nicht nötig.)

**Nützliche Funktionen** (Bibliotheken braucht man keine, außer `random` für die Erweiterung am Ende)
- `dict`, `in`, `list.append`, `len`, `join`

## Die Idee

Ein Zug mit der Augenzahl `z` vom Feld `p` aus läuft in genau dieser Reihenfolge ab:

1. `p` um `z` erhöhen.
2. War das über das Ziel hinaus (`p > 100`)? Dann den Überschuss zurücklaufen: `p = 200 - p` (denn 100 − (p − 100)).
3. Liegt auf dem neuen Feld eine Leiter? Dann zum anderen Ende der Leiter gehen.

Die Reihenfolge ist wichtig: Beim Zurücklaufen vom Ziel kann man auf einer Leiter landen (zum Beispiel von 99 mit einer 3: 102 → 98, und von 98 geht es abwärts nach 92).

Gewürfelt wird immer dieselbe Zahl. Wir zählen die Züge, bis das Feld 100 erreicht ist. Wiederholt sich ein Feld, kommt Frederick nie ins Ziel.

## Schritt 1: Die Leitern

Jede Leiter ist ein Paar `(unten, oben)`. Weil man an beiden Enden ankommen kann und dann zum jeweils anderen Ende muss, tragen wir die Leiter in **beide Richtungen** in ein Wörterbuch ein:
`leitern[6] = 27` und `leitern[27] = 6`.

In [ ]:
PAARE = [(6, 27), (14, 19), (21, 53), (31, 42), (33, 38), (46, 62),
         (51, 59), (57, 96), (65, 85), (68, 80), (70, 76), (92, 98)]

leitern = {}
for a, b in PAARE:
    leitern[a] = b
    leitern[b] = a

print(len(leitern), 'Leiterenden')
print(leitern[6], leitern[27], leitern[98], 98 in leitern, 99 in leitern)

24 Leiterenden
27 6 92 True False


## Schritt 2: Ein Zug

Die Funktion `zug(feld, wurf)` führt die drei Schritte aus und gibt das neue Feld zurück.

In [ ]:
def zug(feld, wurf):
    feld = feld + wurf                  # 1. weiterziehen
    if feld > 100:
        feld = 200 - feld               # 2. über das Ziel hinaus: den Rest zurücklaufen
    if feld in leitern:
        feld = leitern[feld]            # 3. Leiter benutzen
    return feld


print(zug(0, 6))        # Feld 6 ist eine Leiter nach 27
print(zug(27, 4))       # 31 -> Leiter nach 42
print(zug(97, 4))       # 101 -> 99
print(zug(96, 6))       # 102 -> 98 -> Leiter abwärts nach 92

27
42
99
92


## Schritt 3: Immer dieselbe Zahl würfeln

Wir beginnen auf Feld 0 und ziehen so lange, bis das Feld 100 erreicht ist oder ein Feld zum zweiten Mal vorkommt (Endlosschleife).
In der Liste `besucht` stehen alle Felder, auf denen die Figur nach einem Zug stand. Die Funktion gibt zurück, ob das Ziel erreicht wurde,
und die Liste der besuchten Felder (ihre Länge ist die Anzahl der Züge, wenn das Ziel erreicht wurde).

Würde man dagegen einfach „so lange ziehen, bis man im Ziel ist“ programmieren, liefe das Programm bei einer Endlosschleife ewig.

In [ ]:
def spiele(wurf):
    feld = 0
    besucht = []
    while feld != 100:
        feld = zug(feld, wurf)
        if feld in besucht:
            return False, besucht + [feld]       # schon einmal hier gewesen: Endlosschleife
        besucht.append(feld)
    return True, besucht


for wurf in range(1, 7):
    ziel, besucht = spiele(wurf)
    if ziel:
        print(f'Würfelzahl {wurf}: Ziel erreicht nach {len(besucht)} Zügen')
    else:
        print(f'Würfelzahl {wurf}: Endlosschleife, das Ziel wird nie erreicht')

Würfelzahl 1: Ziel erreicht nach 26 Zügen
Würfelzahl 2: Ziel erreicht nach 17 Zügen
Würfelzahl 3: Endlosschleife, das Ziel wird nie erreicht
Würfelzahl 4: Endlosschleife, das Ziel wird nie erreicht
Würfelzahl 5: Ziel erreicht nach 16 Zügen
Würfelzahl 6: Ziel erreicht nach 14 Zügen


Mit der 1, der 2, der 5 und der 6 kommt Frederick ins Ziel, und zwar nach 26, 17, 16 und 14 Zügen. Mit der 3 und der 4 gerät er in eine Endlosschleife.
Das stimmt mit den offiziellen Lösungshinweisen überein.

## Schritt 4: Den Weg ausgeben

Eine reine Ja/Nein-Antwort ist zu wenig. Wir geben den ganzen Weg aus und sagen dazu, wo Leitern benutzt wurden und wo Frederick über das Ziel
hinausgelaufen ist. Dafür baut die Funktion `weg_text` die Beschreibung eines Zuges aus den Zwischenschritten auf:
Zwischenfelder (vor dem Zurücklaufen oder dem Leiterstieg) stehen in runden Klammern.

In [ ]:
def zug_text(feld, wurf):
    text = ''
    neu = feld + wurf
    if neu > 100:
        text = text + f'({neu}) [zu weit] '
        neu = 200 - neu
    if neu in leitern:
        richtung = 'aufwärts' if leitern[neu] > neu else 'abwärts'
        text = text + f'({neu}) [Leiter {richtung}] '
        neu = leitern[neu]
    return text + str(neu)


def zeige_spiel(wurf):
    print(f'Frederick würfelt immer die Zahl {wurf}.')
    ziel, besucht = spiele(wurf)
    feld = 0
    teile = ['[Start]']
    for naechstes in besucht:
        teile.append(zug_text(feld, wurf))
        feld = naechstes
    if ziel:
        print(f'Frederick erreicht das Ziel nach {len(besucht)} Zügen:')
    else:
        print('Frederick verfängt sich in einer Endlosschleife:')
    print(' '.join(teile))
    if not ziel:
        print(f'(Das Feld {besucht[-1]} war schon einmal dran, ab hier wiederholt sich alles.)')
    print()


zeige_spiel(6)

Frederick würfelt immer die Zahl 6.
Frederick erreicht das Ziel nach 14 Zügen:
[Start] (6) [Leiter aufwärts] 27 (33) [Leiter aufwärts] 38 44 50 56 (62) [Leiter abwärts] 46 52 58 64 (70) [Leiter aufwärts] 76 82 88 94 100



## Schritt 5: Alle sechs Zahlen

In [ ]:
for wurf in range(1, 7):
    zeige_spiel(wurf)

Frederick würfelt immer die Zahl 1.
Frederick erreicht das Ziel nach 26 Zügen:
[Start] 1 2 3 4 5 (6) [Leiter aufwärts] 27 28 29 30 (31) [Leiter aufwärts] 42 43 44 45 (46) [Leiter aufwärts] 62 63 64 (65) [Leiter aufwärts] 85 86 87 88 89 90 91 (92) [Leiter aufwärts] 98 99 100

Frederick würfelt immer die Zahl 2.
Frederick erreicht das Ziel nach 17 Zügen:
[Start] 2 4 (6) [Leiter aufwärts] 27 29 (31) [Leiter aufwärts] 42 44 (46) [Leiter aufwärts] 62 64 66 (68) [Leiter aufwärts] 80 82 84 86 88 90 (92) [Leiter aufwärts] 98 100

Frederick würfelt immer die Zahl 3.
Frederick verfängt sich in einer Endlosschleife:
[Start] 3 (6) [Leiter aufwärts] 27 30 (33) [Leiter aufwärts] 38 41 44 47 50 (53) [Leiter abwärts] 21 24 (27) [Leiter abwärts] 6 9 12 15 18 (21) [Leiter aufwärts] 53 56 (59) [Leiter abwärts] 51 54 (57) [Leiter aufwärts] 96 99 (102) [zu weit] (98) [Leiter abwärts] 92 95 (98) [Leiter abwärts] 92
(Das Feld 92 war schon einmal dran, ab hier wiederholt sich alles.)

Frederick würfelt immer 

Bei der **3** läuft Frederick über 96 und 99 nach 102, geht zu weit zurück auf 98 und steigt von dort nach 92 ab. Von 92 geht es 95, 98 (dann wieder abwärts nach 92), 95, 98, ...
im Kreis zwischen 92 und 95 (und dem Zwischenfeld 98). Bei der **4** geht es über 97 hinaus: 97 + 4 = 101 wird zu 99, 99 + 4 = 103 wird zu 97. Er pendelt also
zwischen 97 und 99 hin und her.

## Für Neugierige: Zwei Kontrollen und der Zufall

**Kontrolle 1:** Ohne das Merken der besuchten Felder reicht auch: Nach höchstens 100 Zügen muss Frederick im Ziel sein, sonst hat er ein Feld doppelt betreten.
Wir vergleichen beide Verfahren.

**Kontrolle 2:** Bei der Würfelzahl 1 kommt man immer ins Ziel, egal wie die Leitern liegen (solange jedes Feld höchstens eine Leiter hat), denn man kann nie über das Ziel hinauslaufen.
Probier es mit anderen Leitern aus.

**Zufall:** Frederick würfelt natürlich normalerweise nicht immer dieselbe Zahl. Wie lange dauert ein Spiel mit einem normalen Würfel? Wir spielen 10 000 Spiele
nach und bilden den Durchschnitt. `random.seed(1)` sorgt dafür, dass bei jedem Lauf dasselbe herauskommt.

In [ ]:
import random


def spiele_hoechstens_100(wurf):
    feld = 0
    for nummer in range(1, 101):
        feld = zug(feld, wurf)
        if feld == 100:
            return nummer                # Züge bis zum Ziel
    return None                          # nach 100 Zügen nicht im Ziel: Endlosschleife


for wurf in range(1, 7):
    ziel, besucht = spiele(wurf)
    kurz = spiele_hoechstens_100(wurf)
    print(wurf, kurz, ziel and len(besucht) == kurz)

random.seed(1)
summe = 0
for spiel in range(10000):
    feld = 0
    zuege = 0
    while feld != 100:
        feld = zug(feld, random.randint(1, 6))
        zuege = zuege + 1
    summe = summe + zuege
print('Durchschnitt bei zufälligem Würfeln:', summe / 10000, 'Züge')

1 26 True
2 17 True
3 None False
4 None False
5 16 True
6 14 True
Durchschnitt bei zufälligem Würfeln: 46.109 Züge
